### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [2]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path=Path(".env") if Path(".env").exists() else None)
if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY not found in .env")


/Users/shihab/Downloads/AI/Courses/01Portfolio/LangChainKrisnaik/playground/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
## Step 1: Load and split the dataset
candidate_paths = [
    Path("10. Query Enhancement/2. langchain_crewai_dataset.txt"),
    Path("2. langchain_crewai_dataset.txt"),
]
file_path = next((path for path in candidate_paths if path.exists()), None)
if file_path is None:
    raise FileNotFoundError("Could not find 2. langchain_crewai_dataset.txt")

loader = TextLoader(str(file_path))
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)
chunks


[Document(metadata={'source': '2. langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': '2. langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard patterns like Stuff, Map-Reduce, and Refine. (v1)'),
 Document(metadata={'source': '2. langchain_crewai_dataset

In [4]:
chunks

[Document(metadata={'source': '2. langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': '2. langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using standard patterns like Stuff, Map-Reduce, and Refine. (v1)'),
 Document(metadata={'source': '2. langchain_crewai_dataset

In [5]:
### Step 2: Vector Store
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(chunks, embedding_model)
retriever = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5})
retriever


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x14e75c850>, search_type='mmr', search_kwargs={'k': 5})

In [6]:
## Step 4: LLM and Prompt
llm = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.2)
expansion_prompt = PromptTemplate.from_template("""
Expand the user query into a broader search query for retrieval.
Keep it concise and retrieval-focused.

User query: {query}

Expanded query:
""")
query_expansion_chain = expansion_prompt | llm | StrOutputParser()


In [7]:
# Query expansion
expanded_query = query_expansion_chain.invoke({"query": "LangChain memory"})
print(expanded_query)


"LangChain memory architecture, LangChain memory management, LangChain memory retrieval, LangChain memory storage, LangChain memory access, LangChain memory efficiency, LangChain memory optimization"


In [8]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'"Langchain memory retrieval capabilities"'

In [9]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question using only the retrieved context.

Context:
{context}

Question: {question}
""")


In [10]:
# Step 5: Full RAG pipeline with query expansion
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def expanded_rag_pipeline(user_query):
    expanded = query_expansion_chain.invoke({"query": user_query})
    docs = retriever.invoke(expanded)
    chain = (
        RunnableMap({
            "context": lambda _: format_docs(docs),
            "question": lambda _: user_query,
        })
        | answer_prompt
        | llm
        | StrOutputParser()
    )
    return expanded, docs, chain.invoke({})


In [11]:
# Step 6: Run query
expanded, retrieved_docs, answer = expanded_rag_pipeline("How does LangChain memory work?")
print("Expanded query:\n", expanded)
print("\nAnswer:\n", answer)


Expanded query:
 Expanded query: "LangChain memory architecture, functionality, and data storage mechanisms."

This expanded query captures the core concept of the user's question, allowing for a more comprehensive search for relevant information on LangChain's memory system.

Answer:
 There is no information about LangChain memory in the provided context. The context discusses LangChain's integration with vector databases, compatibility with LLM providers, and prompt engineering capabilities, but it does not mention memory.


In [12]:
# Retrieved docs
retrieved_docs


[Document(id='4809003c-d307-4442-93f9-f665cc957580', metadata={'source': '2. langchain_crewai_dataset.txt'}, page_content='LangChain integrates seamlessly with vector databases like FAISS, Chroma, Pinecone, and Weaviate, enabling semantic search within large document corpora. This capability is especially important in Retrieval-Augmented Generation (RAG), where external knowledge is fetched and injected into the LLM prompt to enhance accuracy and reduce hallucination. (v7)'),
 Document(id='28222210-437c-4a20-b8be-e6718170b9f9', metadata={'source': '2. langchain_crewai_dataset.txt'}, page_content='Prompt engineering is central to LangChain’s design. It provides templating capabilities, input variables, formatting options, and prompt chaining. Developers can reuse prompt templates across different chains and even nest them. (v2)\nLangChain is compatible with multiple LLM providers including OpenAI, Anthropic, Cohere, Hugging Face, and more. This flexibility ensures that developers can sw